# Maximal tuning of the two fingerprint baselines

Exhaustive grid search with stratified 5-fold cross-validation, replacing the
6-candidate randomised search on a single validation split used in the published run.

**Read `README_baseline_tuning.md` in the bundle before running.** It states what the
published protocol actually did, and the four ways this result can change the manuscript.

Attach a **GPU** session: XGBoost uses it, the random forest does not (it is CPU-bound,
so the core count is what matters for that half).


In [ ]:
# Upload gpu_baseline_tuning.tar.gz to the session, then unpack.
import os, glob, subprocess
tb = (glob.glob('/kaggle/input/**/gpu_baseline_tuning.tar.gz', recursive=True)
      + glob.glob('/content/**/gpu_baseline_tuning.tar.gz', recursive=True)
      + glob.glob('./gpu_baseline_tuning.tar.gz'))
assert tb, 'upload gpu_baseline_tuning.tar.gz first'
os.makedirs('/kaggle/working' if os.path.isdir('/kaggle') else '/content/work', exist_ok=True)
WORK = '/kaggle/working' if os.path.isdir('/kaggle') else '/content/work'
subprocess.run(['tar','xzf',tb[0],'-C',WORK], check=True)
os.chdir(os.path.join(WORK,'bl_tune'))
print(sorted(os.listdir('.')))
print('data:', sorted(os.listdir('data')))

In [ ]:
# RDKit and a current xgboost are the only additions most images need.
!pip -q install rdkit-pypi 2>/dev/null || pip -q install rdkit
!python -c "import xgboost,sklearn,rdkit;print('xgboost',xgboost.__version__,'| sklearn',sklearn.__version__)"

## Step 0 — make the checkpoints outlive the session

**Do not skip this.** The grid takes hours. If the runtime is recycled, the laptop
sleeps, or the session limit is reached, everything written to the virtual machine's
own disk is gone. The run keeps a per-configuration log and a per-target prediction
file, and on restart it continues at the first configuration it has no record of —
but only if those files are somewhere that survives.

On **Colab**, mount Drive and point both directories there. On **Kaggle**,
`/kaggle/working` survives a *committed* session, so keep the defaults and use
**Save version** — but note that an interactive session killed before a save loses it,
so commit periodically or prefer Colab with Drive for the long grids.


In [ ]:
import os
ON_KAGGLE = os.path.isdir('/kaggle')
if ON_KAGGLE:
    ROOT = '/kaggle/working/glioma_tuning'
else:
    from google.colab import drive
    drive.mount('/content/drive')           # survives a recycled runtime
    ROOT = '/content/drive/MyDrive/glioma_tuning'
os.environ['GLIOMA_CKPT']  = os.path.join(ROOT, 'checkpoints')
os.environ['GLIOMA_CACHE'] = os.path.join(ROOT, 'cache')
os.makedirs(os.environ['GLIOMA_CKPT'],  exist_ok=True)
os.makedirs(os.environ['GLIOMA_CACHE'], exist_ok=True)
print('checkpoints ->', os.environ['GLIOMA_CKPT'])
print('cache       ->', os.environ['GLIOMA_CACHE'])
print('already checkpointed:', sorted(f for f in os.listdir(os.environ['GLIOMA_CKPT'])
                                      if f.endswith('done.npz')) or 'nothing yet')

## Step 1 — measure before committing

This builds the fingerprint cache once, times one real fit on the largest and smallest
target, and projects the wall time for each grid. If the `--grid full` projection does
not fit the session limit, use `GRID=reduced` in step 2.


In [ ]:
!python smoke_test_baselines.py

## Step 1b — RAPIDS / cuML, only if you want the GPU screening arm

The screening pass ranks a much larger forest grid on the GPU; the winner is then
refit on CPU in scikit-learn, which is what every reported number comes from. Skip
this step entirely if you are running the built-in 81-configuration grid.

On **Kaggle**, select a GPU accelerator and the RAPIDS packages are already in the
image on most GPU images — the check below tells you. On **Colab**, install cuML for
the CUDA 12 runtime. If the install fails or no GPU is attached, do not fake it: run
with `RF_MODE=grid` and say so in the write-up.


In [ ]:
# Is cuML already here?
import subprocess, sys
print(subprocess.run(['nvidia-smi','-L'], capture_output=True, text=True).stdout.strip() or 'no GPU visible')
try:
    import cuml; print('cuml', cuml.__version__, '- nothing to install')
except ImportError:
    print('cuml not present; run the install cell below (Colab) or switch the Kaggle image')

In [ ]:
# Colab only, and only if the check above found no cuml. Takes several minutes.
# Pin nothing here: let pip resolve against the session's CUDA runtime.
!pip -q install --extra-index-url=https://pypi.nvidia.com cuml-cu12
import importlib, cuml; importlib.reload(cuml); print('cuml', cuml.__version__)

In [ ]:
# Which class-weighting route will the screen actually take on THIS cuML build?
# Support is version-dependent: some builds reject only the value
# 'balanced_subsample' (ValueError), others do not know the argument at all
# (TypeError). The probe below runs two-tree fits and reports what it found, so the
# screen never discovers this hours into a run.
import sys, os
sys.path.insert(0, '.')
from screen_rf_cuml import weight_strategy
kind, note = weight_strategy('cuml', 0)
print('strategy the screen will use :', kind)
print('reason                       :', note)
if kind == 'none':
    print('\n-> The screen will run UNWEIGHTED. It is still usable as a ranking device,')
    print('   but read screening_gain in the confirmation output before trusting it.')
print('\nThe confirmation stage is unaffected either way: it refits in scikit-learn')
print('with class_weight="balanced_subsample", which is the published setting.')


## Step 2 — the run

`GRID=full` is the exhaustive search. `SEEDS='0'` tunes one seed only, which still
answers the question if time is short — the published run tuned only seed 0 too.
Output JSONs are skipped if present, so a restarted session resumes.


In [ ]:
# Two ways to run. Use ONE of them.
#
# (A) %%bash cell magic: `%%bash` must be the FIRST line of the cell, alone, with the
#     exports and the command below it. Nothing may precede it in that cell.
#
# (B) `!` shell escape, which is what you are using. Each `!` line is its own
#     subshell, so `!export RF_MODE=screen` on a line by itself is forgotten the
#     moment that line ends. Two things that DO work with `!`:
#       - set the variables in the kernel environment first, as this cell does,
#         because `!` inherits os.environ; or
#       - prefix them on the same line:  !RF_MODE=screen bash run_tuning.sh
import os
os.environ['GRID']        = 'full'    # CPU grid for XGBoost, and for RF when RF_MODE=grid
os.environ['SEEDS']       = '0'       # one seed is ~3.8 h on a T4 session; '0 1 2' is ~11.4 h
os.environ['FOLDS']       = '5'
os.environ['RF_MODE']     = 'screen'  # 'grid' = the 81-config CPU grid, no GPU needed
os.environ['SCREEN_GRID'] = 'medium'  # small=96, medium=320, large=960 configurations
os.environ['TOPK']        = '24'
# GLIOMA_CKPT / GLIOMA_CACHE were set in step 0 and are inherited the same way.
for k in ['GRID','SEEDS','FOLDS','RF_MODE','SCREEN_GRID','TOPK','GLIOMA_CKPT','GLIOMA_CACHE']:
    print(f'{k:14s} = {os.environ.get(k, "(unset)")}')


In [ ]:
!bash run_tuning.sh


Rehearsal first, on a new platform: `!ONLY_TASK=CDK6 GRID=reduced SEEDS=0 bash run_tuning.sh`
finishes in about ten minutes and exercises the whole path, including the checkpoint
writes and the external-set scoring.


## Step 3 — archive before the session dies

`metrics/` holds the CV tables and the external-set predictions. A lost session cannot
be reconstructed, so download this file before doing anything else.


In [ ]:
!tar czf tuned_baselines.tar.gz metrics logs
import os; print('%.1f MB' % (os.path.getsize('tuned_baselines.tar.gz')/1e6))
# The checkpoint directory is the crash-recovery copy; this tarball is the deliverable.
try:
    from google.colab import files; files.download('tuned_baselines.tar.gz')
except Exception:
    print('Kaggle: the file is in /kaggle/working, use the Output tab to download it.')


In [ ]:
# Quick look at what was chosen, before you hand the tarball back.
import json, glob, pandas as pd
for f in sorted(glob.glob('metrics/tuned_*_s*.json')):
    d = json.load(open(f))
    print('==', f, '| configurations searched:', d['n_configurations'],
          '| folds:', d['folds'], '| %.0f s' % d['wall_seconds'])
    print(pd.DataFrame([{'target': t, 'cv_roc_auc': round(v['cv_roc_auc_mean'], 4),
                         **v['params']} for t, v in d['chosen'].items()]).to_string(index=False))